# Cambodia CPI: AI COICOP Classification for Unclassified / New Products

This notebook demonstrates how our pipeline uses the **Google Gemini 2.5 Flash API** to classify newly scraped or unclassified products into the official **UN / Cambodia National Institute of Statistics (NIS) 4-digit COICOP Taxonomy**.

### Key Capabilities Demonstrated:
1. **Querying Unclassified Products**: Pulls products directly from PostgreSQL (`cpi_db`) that lack 4-digit classification.
2. **40x Batch Packaging**: Sends products in batches of 40 per API call to maximize throughput and stay within free-tier limits.
3. **Domain Guardrails**: Enforces hard rules in the system prompt (e.g. Pet Food -> 09.3.1, Personal Care -> 12.1.3).
4. **Classify Once, Keep Forever**: Writes the resulting 4-digit UN codes permanently into `silver.canonical_items`.

## 1. Connect to PostgreSQL (`cpi_db`)

In [ ]:
import os
import json
import psycopg2
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv(dotenv_path="../.env")

# Connect to PostgreSQL cpi_db
conn = psycopg2.connect(
    dbname=os.getenv("CPI_DB_NAME", "cpi_db"),
    user=os.getenv("CPI_DB_USER", "cpi_user"),
    password=os.getenv("CPI_DB_PASSWORD", "cpi_pass"),
    host="localhost",
    port=int(os.getenv("POSTGRES_PORT", 5432))
)
print("Connected to PostgreSQL cpi_db successfully!")

## 2. Inspect Products Pending 4-Digit Classification
We query products in `silver.canonical_items` that have generic `*.unclassified` codes.

In [ ]:
query = """
SELECT item_id, canonical_name, brand, coicop_division, coicop_code
FROM silver.canonical_items
WHERE coicop_code LIKE '%.unclassified' OR coicop_code IS NULL
ORDER BY item_id
LIMIT 10;
"""
df_pending = pd.read_sql(query, conn)
df_pending

## 3. Initialize Gemini AI Client with Key Pool

In [ ]:
import sys
sys.path.append("..")

from google import genai
from google.genai import types as genai_types
from pipeline.key_pool import get_key_pool

key_pool = get_key_pool()
api_key = key_pool.get_next_key()
client = genai.Client(api_key=api_key)
print(f"Gemini Client initialized! Key pool count: {key_pool.get_key_count()}")

## 4. Classify Batch using Gemini 2.5 Flash with Domain Guardrails

In [ ]:
SYSTEM_PROMPT = """You are an expert UN COICOP classification engine for the Consumer Price Index (CPI) in Cambodia.
Classify consumer products into the official 4-digit NIS Cambodia COICOP Class taxonomy (DD.G.C, e.g. 01.1.1, 01.1.4, 05.6.1, 12.1.3).

CRITICAL GUARDRAIL RULES:
- Pet food & accessories: MUST be '09.3.1' (Pets), NEVER Division 01.
- Personal care & soaps: MUST be '12.1.3' (Personal Care), NEVER Division 01 or 05.
- Household cleaning & detergents: MUST be '05.6.1' (Non-durable household goods).
- Milk, dairy, yogurt: MUST be '01.1.4'.
- Bread, rice, cereals: MUST be '01.1.1'.
- Confectionery, chocolates & snacks: MUST be '01.1.8'.
- Camping chairs & furniture: MUST be '05.1.1'.

Return strict JSON list of objects: [{"id": "<item_id>", "coicop_division": "<DD>", "coicop_code": "<DD.G.C>", "reason": "<short justification>"}]
"""

# Format products into batch JSON payload
batch_payload = [
    {"id": str(row["item_id"]), "name": row["canonical_name"]}
    for _, row in df_pending.iterrows()
]

prompt_text = "Classify the following products:\n" + json.dumps(batch_payload, ensure_ascii=False, indent=2)

# Call Gemini 2.5 Flash
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_text,
    config=genai_types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
        temperature=0.0
    )
)

# Parse JSON output
ai_classifications = json.loads(response.text)
df_classified = pd.DataFrame(ai_classifications)

# Merge to see Before vs After AI Classification
df_comparison = df_pending.merge(df_classified, left_on="item_id", right_on="id")
df_comparison[["canonical_name", "coicop_code_x", "coicop_division_y", "coicop_code_y", "reason"]].rename(
    columns={
        "coicop_code_x": "before_code",
        "coicop_division_y": "ai_division",
        "coicop_code_y": "ai_4digit_code"
    }
)

## 5. Permanently Persist Results to `silver.canonical_items`
Once verified, we update the master database table so these products are permanently saved and never need AI classification again.

In [ ]:
with conn.cursor() as cur:
    for item in ai_classifications:
        cur.execute("""
            UPDATE silver.canonical_items
            SET coicop_division = %s,
                coicop_code = %s
            WHERE item_id = %s;
        """, (item["coicop_division"], item["coicop_code"], item["id"]))
    conn.commit()

print(f"Successfully updated {len(ai_classifications)} canonical items in PostgreSQL!")